# EutranCellRelation 4G--4G

##### O MO EUtranCellRelation contem relações 4G--4G e 5G--4G. Nesse caso estou separando as relações 4G--4G
##### 1 - Separar apenas as relações 4G--4G
##### 2 -  Separar o EUtranCellRelationId com cellname (parte2) e EUtranCellRelationId com GGI (parte1).
##### 3 - Fazer o join do parte 1 com o arquivo EUtranCellFDD para identificar o cellname do "EUtranCellRelationId"
##### 4 - Concatenar com o parte2 

In [ ]:
import pandas as pd

chunk_size = 50000  # You can adjust this size based on your available memory
chunks = []

# neighborCellRef traz o MO da vizinha e e a unica fonte confiavel do alvo da relacao.
for chunk in pd.read_csv('C:/Dumps/LTE_total/EUtranCellRelation.txt', sep='\t',
                         usecols=['MO','UF','eNB','CELL','Freq_tgt', 'EUtranFreqRelation','EUtranCellRelationId', 'sCellCandidate', 'sCellPriority', 'coverageIndicator', 'ManagedElement', 'neighborCellRef'],
                         chunksize=chunk_size, low_memory=False):
    chunks.append(chunk)

# Combine chunks into a single DataFrame
df = pd.concat(chunks, ignore_index=True)

# Algumas relações vêm sem UF/eNB, mas trazem o eNB de origem no ManagedElement.
df['eNB'] = df['eNB'].replace('', pd.NA).fillna(df['ManagedElement'])
df['UF'] = df['UF'].replace('', pd.NA).fillna(df['eNB'].str[-2:])
df.drop(columns='ManagedElement', inplace=True)

pd.set_option('display.max_colwidth', None)

df.head()

In [ ]:
# Monta a linha de comando a partir dos campos disponíveis no novo parser XML.
# Se a coluna MO não existir ou não estiver preenchida, usamos EUtranFreqRelation e EUtranCellRelationId.

def extrair_linha_comando(row):
    if pd.isna(row['CELL']) or pd.isna(row['EUtranFreqRelation']) or pd.isna(row['EUtranCellRelationId']):
        return ''

    chave = 'EUtranCellTDD' if str(row['CELL']).startswith(('L', 'O')) else 'EUtranCellFDD'
    return f"{chave}={row['CELL']},EUtranFreqRelation={row['EUtranFreqRelation']},EUtranCellRelation={row['EUtranCellRelationId']}"

# Se ainda existir a coluna MO com dados, priorizamos manter o mesmo formato do comando original.
# Caso contrário, construímos manualmente a linha com as colunas disponíveis.

df['linha_comando'] = df.apply(extrair_linha_comando, axis=1)

df.sample(2)

In [ ]:
# O alvo da relacao sai SEMPRE do MO apontado em neighborCellRef, nunca do
# EUtranCellRelationId. O Id e apenas um rotulo: costuma ser o CGI quando a vizinha e
# externa e o nome da celula quando e interna, mas existem relacoes para celula EXTERNA
# batizadas com nome de celula. Deduzir o eNB de destino pelo primeiro caractere do Id
# fazia essas relacoes herdarem o eNB de ORIGEM, e a comparacao de chave de eNB passava
# a confrontar o eNB consigo mesmo - o que tornava 'inter-site' inalcancavel para elas.
referencia = df['neighborCellRef'].fillna('')

# Vizinha externa: o MO traz o CGI, e a celula vive em outro eNB.
df['CGI_alvo'] = referencia.str.extract(r'vsDataExternalEUtranCell(?:FDD|TDD)=([^,]+)', expand=False)
# Vizinha interna: o MO traz o nome da celula, no proprio eNB de origem.
df['CELL_alvo_interna'] = referencia.str.extract(r'vsDataEUtranCell(?:FDD|TDD)=([^,]+)', expand=False)

# parte1: alvo resolvido pelo CGI contra o inventario (feito na celula do EUtranCellFDD).
parte1 = df[df['CGI_alvo'].notna()].copy()

# parte2: alvo interno, entao aqui sim eNB_tgt = eNB e verdade por definicao do MO.
parte2 = df[df['CGI_alvo'].isna() & df['CELL_alvo_interna'].notna()].copy()
parte2['eNB_tgt'] = parte2['eNB']
parte2['CELL_tgt'] = parte2['CELL_alvo_interna']

# Ficam fora do audit, mas nao somem em silencio: vao para a aba 'nao_auditadas'.
sem_referencia = df[df['CGI_alvo'].isna() & df['CELL_alvo_interna'].isna()].copy()

print(f'relacoes no dump          : {len(df):,}')
print(f'  alvo externo (via CGI)  : {len(parte1):,}')
print(f'  alvo interno (mesmo eNB): {len(parte2):,}')
print(f'  sem neighborCellRef     : {len(sem_referencia):,}')

parte2.sample(5)

# EUtranCell - bloco de scell usado para o scellPriority

In [ ]:
#arquivo_EUtranCellFDD= 'C:/Camilo/' + 'EUtranCellFDD_TDD_FDD'+ '_'+str(semana_atual)+'.csv'

EUtranCellFDD= pd.read_csv('C:/Camilo/EUtranCell_TDD_FDD.csv', usecols= ['CN','MUNICIPIO','eNB','eNBId','CELL', 'CGI', 'earfcndl'], engine='python'
                           , encoding='ISO-8859-1')

# Criando o EUtranCell que sera usado para o scellPriority
EUtranCell= EUtranCellFDD.copy()

EUtranCellFDD.rename(columns={'eNB': 'eNB_tgt', 'eNBId': 'eNBId_tgt','CELL': 'CELL_tgt', 'earfcndl': 'earfcndl_tgt'},inplace=True)

# O CGI vem de CGI_alvo (extraido do neighborCellRef), nao do EUtranCellRelationId.
junta= pd.merge(parte1, EUtranCellFDD, right_on= 'CGI', left_on= 'CGI_alvo', how='left')

junta.head()

In [ ]:
junta2= pd.merge(parte2, EUtranCellFDD, right_on= ['eNB_tgt', 'CELL_tgt'], left_on= ['eNB_tgt', 'CELL_tgt'],how='left')

junta2.head()

# Junta 3 é usado para o sCellPriority

# Audit `sCellCandidate` — regras de negócio

> Esta seção é a referência funcional do audit. Se a regra mudar, atualizar esta documentação e a célula de código imediatamente abaixo na mesma alteração.

## Como o alvo da relação é identificado

O alvo sai **sempre** do MO apontado em `neighborCellRef`, nunca do `EUtranCellRelationId`:

- `vsDataExternalEUtranCellFDD=<CGI>` → vizinha **externa**, em outro eNB. `eNB_tgt` e `CELL_tgt` são resolvidos pelo CGI contra o `EUtranCell_TDD_FDD.csv`.
- `vsDataEUtranCellFDD=<nome>` → vizinha **interna**, no próprio eNB. Só neste caso `eNB_tgt = eNB`.

O `EUtranCellRelationId` é apenas um rótulo e **não** indica onde a vizinha mora: existem relações para célula externa batizadas com nome de célula. Deduzir o destino pelo primeiro caractere do Id fazia essas relações herdarem o eNB de origem — e, junto com ele, a UF de origem.

## A chave de site: UF + MID(CELL, 2, 3)

A **UF entra sempre**, porque o código de 3 letras se repete entre estados diferentes e sem ela dois sites distintos virariam o mesmo. Em Python, `MID(nome, 2, 3)` é `nome.str[1:4]`:

```
chave_CELL_src = UF de origem  + CELL_src[1:4]
chave_CELL_tgt = UF de destino + CELL_tgt[1:4]
```

A UF de origem vem da própria relação; a de destino, do sufixo do `eNB_tgt` — que só é confiável depois da resolução pelo `neighborCellRef`. Em relação entre UFs vizinhas, o `eNB_tgt` herdado da origem carimbava a UF errada no destino.

O **site físico vem sempre do nome da célula**, nunca do host — mesma regra de ouro do CellRef. `UF + MID(CELL,2,3)` igual na origem e no destino é a premissa de co-site, e só isso decide a classificação.

## Classificação e valores esperados

| `Tipo` gravado no relatório | Condição | `sCellCandidate` esperado | `coverageIndicator` esperado |
|---|---|---|---|
| `co-setor` | Chave CELL origem = destino e último caractere da CELL origem = destino | `ALLOWED` (1) | `OVERLAP` (1) |
| `co-site intra-eNodeB` | Chave CELL origem = destino, mas os últimos caracteres das CELLs são diferentes. Exemplo: `TABE11 -> TABE12` | `ONLY_ALLOWED_FOR_DL` (3) | `OVERLAP` (1) |
| `inter-site` | Chave CELL origem != destino | `AUTO` (2) | `NONE` (0) |

## O eNB não entra na classificação

Duas consequências, ambas intencionais:

**Co-site é co-site, mesmo em nós diferentes.** Uma torre pode ter dois eNBs — no site `LEK` em Luís Eduardo Magalhães (BA), o `TLEK1_BA` tem a camada 3350 e o `MLEK2_BA` tem 525/1300/9510, mesma coordenada e mesmos azimutes 60/180/300. Como `UF + MID(CELL,2,3)` é o mesmo, essas 96.063 relações são co-site (ou co-setor) como quaisquer outras e seguem os mesmos valores esperados. O sufixo `intra-eNodeB` no nome do `Tipo` é herança do relatório antigo e não afirma que o nó seja o mesmo.

**Hotel de BTS não é exceção.** Um eNB hospedeiro pode servir células de sites remotos distintos — o `MECA2_BA` (site `BA|ECA`) carrega células `TCA1251` do site `BA|CA1` e `TCA2251` do site `BA|CA2`. As chaves de CELL diferem, então a relação é `inter-site` como qualquer outra: compartilhar baseband não coloca as torres no mesmo lugar. Até 03/08/2026 havia uma exceção `Hotel de BTS - co-site inter-eNodeB` esperando `ALLOWED`/`OVERLAP` nesses casos; foi removida por usar o host para decidir o site e por divergir da prática — 61% dessas 3.402 relações já estavam em `AUTO`/`NONE` na rede.

## Pontos importantes

- Para co-setor e co-site, somente `OVERLAP` é válido; `COVERS` deve ser reportado como erro.
- Para inter-site, `coverageIndicator=0`, `0 (NONE)` e `NONE` são equivalentes.
- O código aceita os valores textuais e numéricos retornados pelo parser, por exemplo `ALLOWED`, `1` e `1 (ALLOWED)`.
- A feature de load balancing não é auditada diretamente; este relatório audita os valores finais de `sCellCandidate` e `coverageIndicator` conforme a tabela.
- O Excel contém somente relações inconsistentes, em três abas: `co_site`, `inter_site` e `nao_auditadas` (relações sem `neighborCellRef` utilizável ou com CGI ausente do inventário).

In [ ]:
junta3= pd.concat([junta, junta2], axis=0)
#junta3['CELL_tgt'].str[:1].value_counts()

# Vizinha externa cujo CGI nao existe no EUtranCell_TDD_FDD: sem o alvo nao da para
# classificar. Sai do audit, mas e preservada para a aba 'nao_auditadas'.
nao_resolvidas = junta3[junta3['CELL_tgt'].isna()].copy()

junta3.dropna(subset='CELL_tgt', inplace=True)

junta3.drop(columns='MO', inplace=True)

# Chaves de site: UF + MID(nome, 2, 3).
# A UF de destino e obtida do sufixo do eNB de destino.
junta3['UF_src_chave'] = junta3['UF'].fillna('').astype(str).str.strip().str.upper()
junta3['UF_tgt_chave'] = junta3['eNB_tgt'].fillna('').astype(str).str.strip().str[-2:].str.upper()
junta3['chave_CELL_src'] = junta3['UF_src_chave'] + junta3['CELL'].fillna('').astype(str).str[1:4].str.upper()
junta3['chave_CELL_tgt'] = junta3['UF_tgt_chave'] + junta3['CELL_tgt'].fillna('').astype(str).str[1:4].str.upper()
# As chaves de eNB ficam so como informacao. Elas NAO classificam nada: MID(eNB,2,3)
# descarta a letra da banda e o numero do no, entao dois eNBs da mesma torre
# (TLEK1_BA e MLEK2_BA) produzem a mesma chave e sao indistinguiveis aqui.
junta3['chave_eNB_src'] = junta3['UF_src_chave'] + junta3['eNB'].fillna('').astype(str).str[1:4].str.upper()
junta3['chave_eNB_tgt'] = junta3['UF_tgt_chave'] + junta3['eNB_tgt'].fillna('').astype(str).str[1:4].str.upper()

priority= junta3.copy()

# Classificacao pela regra de ouro: o site fisico vem do NOME DA CELULA, nunca do host.
# Chave de CELL igual -> mesma torre (co-setor ou co-site); diferente -> inter-site.
# Nao ha excecao de Hotel de BTS: celulas de sites remotos distintos no mesmo hospedeiro
# continuam sendo sites diferentes, logo inter-site.
mesma_chave_cell = junta3['chave_CELL_src'] == junta3['chave_CELL_tgt']
mesmo_setor = junta3['CELL'].fillna('').astype(str).str[-1:] == junta3['CELL_tgt'].fillna('').astype(str).str[-1:]

junta3['Tipo'] = 'inter-site'
junta3.loc[mesma_chave_cell & ~mesmo_setor, 'Tipo'] = 'co-site intra-eNodeB'
junta3.loc[mesma_chave_cell & mesmo_setor, 'Tipo'] = 'co-setor'

print(junta3['Tipo'].value_counts().to_string())

# Valores aceitos pelo parser nos formatos textual e numerico.
valores_scellcandidate_allowed = {'ALLOWED', '1', '1 (ALLOWED)'}
valores_scellcandidate_only_dl = {'ONLY_ALLOWED_FOR_DL', '3', '3 (ONLY_ALLOWED_FOR_DL)'}
valores_scellcandidate_auto = {'AUTO', '2', '2 (AUTO)'}
valores_coverageindicator_zero = {'0', '0 (NONE)', 'NONE'}
valores_coverageindicator_overlap = {'OVERLAP', '1', '1 (OVERLAP)'}

# Audit inter-site: esperado sCellCandidate=AUTO e coverageIndicator=0/NONE.
inter_site_df = junta3[junta3['Tipo'] == 'inter-site'].copy()
inter_site_df['sCellCandidate_esperado'] = 'AUTO (2)'
inter_site_df['coverageIndicator_esperado'] = 'NONE (0)'
inter_site_df['Teste_sCellCandidate_inter_site'] = (~inter_site_df['sCellCandidate'].fillna('').astype(str).str.strip().str.upper().isin(valores_scellcandidate_auto)).astype(int).astype(str)
inter_site_df['Teste_coverageIndicator_inter_site'] = (~inter_site_df['coverageIndicator'].fillna('').astype(str).str.strip().str.upper().isin(valores_coverageindicator_zero)).astype(int).astype(str)

inter_site_df = inter_site_df.loc[
    (inter_site_df['Teste_sCellCandidate_inter_site'] == '1') |
    (inter_site_df['Teste_coverageIndicator_inter_site'] == '1')
]

# Audit co-setor e co-site intra-eNodeB.
novo_df = junta3[junta3['Tipo'] != 'inter-site'].copy()
novo_df['sCellCandidate_normalizado'] = novo_df['sCellCandidate'].fillna('').astype(str).str.strip().str.upper()
novo_df['sCellCandidate_esperado'] = 'ALLOWED (1)'
mascara_intra_enodeb = novo_df['Tipo'] == 'co-site intra-eNodeB'
novo_df.loc[mascara_intra_enodeb, 'sCellCandidate_esperado'] = 'ONLY_ALLOWED_FOR_DL (3)'
novo_df['coverageIndicator_esperado'] = 'OVERLAP (1)'

novo_df['Teste_sCellCandidate'] = '0'
novo_df.loc[~mascara_intra_enodeb & ~novo_df['sCellCandidate_normalizado'].isin(valores_scellcandidate_allowed), 'Teste_sCellCandidate'] = '1'
novo_df.loc[mascara_intra_enodeb & ~novo_df['sCellCandidate_normalizado'].isin(valores_scellcandidate_only_dl), 'Teste_sCellCandidate'] = '1'

# Para co-setor e co-site, coverageIndicator esperado e OVERLAP.
novo_df['Teste_coverageIndicator'] = (~novo_df['coverageIndicator'].fillna('').astype(str).str.strip().str.upper().isin(valores_coverageindicator_overlap)).astype(int).astype(str)

# Somente inconsistencias seguem para o relatorio.
filtrado_df = novo_df.loc[(novo_df['Teste_sCellCandidate'] == '1') | (novo_df['Teste_coverageIndicator'] == '1')]
filtrado_df.head()

In [ ]:
EUtranCellFDD.rename(columns={'eNB_tgt':'eNB', 'CELL_tgt': 'CELL','earfcndl_tgt': 'earfcndl'},inplace=True)

EUtranCellFDD.drop(columns=['MUNICIPIO', 'CGI'], inplace=True)

filtrado_df1= pd.merge(filtrado_df, EUtranCellFDD, on=['eNB', 'CELL'], how='left')

filtrado_df1.rename(columns={'eNB':'eNB_src', 'CELL':'CELL_src','earfcndl': 'earfcndl_src'}, inplace=True)

filtrado_df1['Regional']= filtrado_df1['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'],['N', 'N', 'N', 'N', 'N', 'BASE','BASE'])

# Falta apenas organizar as colunas de acordo com a macro
filtrado_df1= filtrado_df1[['Regional','UF','MUNICIPIO','eNB_src','CELL_src','earfcndl_src','Freq_tgt','EUtranCellRelationId','sCellCandidate','sCellCandidate_esperado', 'coverageIndicator','coverageIndicator_esperado', 'linha_comando', 'eNB_tgt',
                            'CELL_tgt','earfcndl_tgt','CGI','Teste_sCellCandidate','Teste_coverageIndicator','Tipo']]

filtrado_df1 = filtrado_df1.sort_values(['Regional', 'UF', 'MUNICIPIO', 'eNB_src', 'CELL_src', 'eNB_tgt', 'CELL_tgt'], ascending=[True, True, True, True, True, True, True])

inter_site_df1= pd.merge(inter_site_df, EUtranCellFDD, on=['eNB', 'CELL'], how='left')

inter_site_df1.rename(columns={'eNB':'eNB_src', 'CELL':'CELL_src','earfcndl': 'earfcndl_src'}, inplace=True)

inter_site_df1['Regional']= inter_site_df1['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'],['N', 'N', 'N', 'N', 'N', 'BASE','BASE'])

inter_site_df1= inter_site_df1[['Regional','UF','MUNICIPIO','eNB_src','CELL_src','earfcndl_src','Freq_tgt','EUtranCellRelationId','sCellCandidate','sCellCandidate_esperado', 'coverageIndicator','coverageIndicator_esperado', 'linha_comando', 'eNB_tgt',
                                'CELL_tgt','earfcndl_tgt','CGI','Teste_sCellCandidate_inter_site','Teste_coverageIndicator_inter_site','Tipo']]

inter_site_df1 = inter_site_df1.sort_values(['Regional', 'UF', 'MUNICIPIO', 'eNB_src', 'CELL_src', 'eNB_tgt', 'CELL_tgt'], ascending=[True, True, True, True, True, True, True])

# Relacoes que o audit nao conseguiu avaliar. Ficam visiveis em vez de sumir no dropna.
nao_auditadas = pd.concat([
    nao_resolvidas.assign(Motivo='CGI da vizinha externa nao existe no EUtranCell_TDD_FDD'),
    sem_referencia.assign(Motivo='relacao sem neighborCellRef utilizavel'),
], axis=0, ignore_index=True)

nao_auditadas = nao_auditadas.reindex(columns=['UF','eNB','CELL','EUtranCellRelationId','Freq_tgt',
                                              'CGI_alvo','sCellCandidate','coverageIndicator',
                                              'linha_comando','Motivo'])
nao_auditadas = nao_auditadas.sort_values(['Motivo','UF','eNB','CELL'])

with pd.ExcelWriter('C:/Vivo Reports/LTE/LTE Mobilidade/Audit_sCellCandidate.xlsx') as writer:
    filtrado_df1.to_excel(writer, sheet_name='co_site', index=False)
    inter_site_df1.to_excel(writer, sheet_name='inter_site', index=False)
    nao_auditadas.to_excel(writer, sheet_name='nao_auditadas', index=False)

print(f'co_site       : {len(filtrado_df1):,}')
print(f'inter_site    : {len(inter_site_df1):,}')
print(f'nao_auditadas : {len(nao_auditadas):,}')
print('Done')

# Audit sCellPriority
#### Usando parte da estrutura do audit de sCellCandidate para o audit de sCellPriority

EUtranCell['CELL_concatenada']= EUtranCell['CELL'].str[1:4]

EUtranCell1= EUtranCell[['CN', 'MUNICIPIO','CELL_concatenada']]

EUtranCell1= EUtranCell1.copy()

EUtranCell1.drop_duplicates(subset=['CELL_concatenada'],inplace=True)
EUtranCell1.rename(columns={'CN':'CN_src', 'MUNICIPIO': 'MUNICIPIO_src'},inplace=True)
EUtranCell1.head()

# Verifica a condição
priority['Tipo'] = priority['CELL_concatenada'] == priority['CELL_tgt_concatenada']

# priority= priority[(priority['UF']=='SP') & (priority['MUNICIPIO'].isin(['SÃO BERNARDO DO CAMPO', 'GUARULHOS', 'JUNDIAÍ']))]

# priority= priority[priority['UF']=='MG']
priority= priority[priority['UF'].isin(['AM', 'AP', 'MA', 'PA', 'RR'])]
priority.head()

priority1= priority.merge(EUtranCell1, on='CELL_concatenada', how='left')

# Primeiro, obtemos a lista de colunas
cols = list(priority1.columns)

# Removemos 'CN_src' e 'MUNICIPIO_src' da lista
cols.remove('CN_src')
cols.remove('MUNICIPIO_src')

# Agora, inserimos 'CN_src' e 'MUNICIPIO_src' nas posições desejadas
# Lembre-se que em Python, a indexação começa em 0, então a posição 1 é a segunda posição
cols.insert(1, 'CN_src')
cols.insert(2, 'MUNICIPIO_src')

# Reordenamos o DataFrame
priority1 = priority1[cols]
priority1.rename(columns={'CN': 'CN_tgt', 'MUNICIPIO': 'MUNICIPIO_tgt'},inplace=True)
priority1.head()

# Priority_false contempla apenas vizinhos externos.
priority_false= priority1.copy()
priority_false= priority_false[(priority_false['Tipo']==False) & (priority_false['sCellPriority']!=5) & (priority_false['CELL_tgt'].str[:1]!='Z') | \
                         (priority_false['Tipo']==False) & (priority_false['sCellPriority']!=4) & (priority_false['CELL_tgt'].str[:1]=='Z')]

# priority_false['sCellPriority'].value_counts()

# priority_false['sCellPriority'].value_counts()
priority_false['Tipo'].replace(False, 'vizinha_externa', inplace=True)
priority_false.head()

# Priority_false contempla vizinhos co-site e co-setores.

priority_true= priority1.copy()

priority_true= priority_true[priority_true['Tipo']==True]

priority_true['Tipo']= priority_true['CELL'].str[-1:]==priority_true['CELL_tgt'].str[-1:]

priority_true['Tipo']= priority_true['Tipo'].replace([True, False], ['co_setor', 'co_site'])

priority_co_setor= priority_true[(priority_true['Tipo']=='co_setor') & (priority_true['sCellPriority']!=7) & (priority_true['CELL_tgt'].str[:1]!='Z') | \
                         (priority_true['Tipo']=='co_setor') & (priority_true['sCellPriority']!=6) & (priority_true['CELL_tgt'].str[:1]=='Z')]

priority_co_site= priority_true[(priority_true['Tipo']=='co_site') & (priority_true['sCellPriority']!=5) & (priority_true['CELL_tgt'].str[:1]!='Z') | \
                         (priority_true['Tipo']=='co_site') & (priority_true['sCellPriority']!=4) & (priority_true['CELL_tgt'].str[:1]=='Z')]

# priority_co_site[['earfcndl_tgt', 'sCellPriority']].value_counts()

priority.shape

audit_priority= pd.concat([priority_false, priority_co_site, priority_co_setor], axis=0)
audit_priority.head()
audit_priority.to_csv('C:/Camilao/sCellPriority_N.csv', index=None)

CN11 = audit_priority[(audit_priority['CN_src'] == 11.0) & (audit_priority['MUNICIPIO_src'] != 'SÃO PAULO')]
CN11.to_csv('C:/Camilao/sCellPriority_CN11_sem_SP.csv', index=None)

SP = audit_priority[(audit_priority['CN_src'] == 11.0) & (audit_priority['MUNICIPIO_src'] == 'SÃO PAULO')]

import pandas as pd

# Calcular o número de linhas para dividir o DataFrame
num_linhas = len(SP) // 2

# Dividir o DataFrame em duas partes
SP_parte1 = SP.iloc[:num_linhas, :]
SP_parte2 = SP.iloc[num_linhas:, :]

# Salvar cada parte em um arquivo CSV
SP_parte1.to_csv('C:/Vivo Reports/LTE/LTE Mobilidade/sCellPriority_SP_parte1.csv', index=False)
SP_parte2.to_csv('C:/Vivo Reports/LTE/LTE Mobilidade/sCellPriority_SP_parte2.csv', index=False)
print('sCellPriority, Done')

In [ ]:
# audit_priority.to_excel('C:/Camilao/Audit_sCellPriority.xlsx', index=None)
# print('Done')

In [ ]:
"""

priority['Regional']= priority['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

priority.head()

# Realizando a contagem dos valores distintos de 'sCellPriority' por 'Regional'
contagem_distintos = priority.groupby(['Regional', 'sCellPriority']).size().reset_index(name='count')


# Pivotando o DataFrame para obter 'sCellPriority' como linhas e 'Regional' como colunas
pivot_df = contagem_distintos.pivot(index='sCellPriority', columns='Regional', values='count')

# Renomeando o índice para 'Value'
pivot_df = pivot_df.rename_axis('Value').reset_index()

# Exibindo o DataFrame pivotado
pivot_df.to_excel('C:/Camilao/scellpriority_Higa.xlsx', index=None)

"""

In [ ]:
"""
nulos = priority['sCellPriority'].isna().sum()
nulos
"""

In [ ]:
"""
priority[['Regional','sCellPriority']].value_counts().reset_index()
"""